<a href="https://colab.research.google.com/github/brianback-seungho/bus5_app/blob/main/Overflight_Permit_Automation_Colab_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 대한민국 영공통과 허가 신청서 자동 점검기 v2\n\n- PDF 신청서를 로컬에서 업로드합니다.\n- 코드에 내장된 항공로 기준표로 진입지점/항공로/진출지점을 검증합니다.\n- 날짜와 요일, 위험물 No 체크 여부를 검증합니다.\n- E-MAIL 도메인이 `jetex.com`이면 Applicant를 `JETEX FLIGHT SUPPORT`로 출력합니다.\n- 검증 통과 시 `허가신청템플릿.txt`를 생성하고 다운로드합니다.\n

In [ ]:
!pip -q install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 63.6 MB/s eta 0:00:00


In [6]:
# -*- coding: utf-8 -*-
"""
대한민국 영공통과 허가 신청서 자동 점검 및 텍스트 템플릿 생성기 v2l
- Colab 실행용
- PDF 신청서를 업로드하면 날짜/요일, 항공로, 위험물 체크를 검증합니다.
- 검증 통과 시 영공통과 허가 신청서(Aircraft identification).txt 형식의 파일을 생성합니다.
- 검증 실패 시 검증오류보고서.txt 파일을 생성합니다.

사용법(Colab):
1) !pip -q install PyMuPDF
2) 이 파일 내용을 셀에 붙여넣고 실행하거나, .py로 업로드 후 실행
3) 파일 선택 버튼에서 PDF 신청서를 선택
"""

import os
import re
import sys
import subprocess
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

try:
    import fitz  # PyMuPDF
except ImportError:  # Colab에서 직접 .py 실행 시 자동 설치
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "PyMuPDF"])
    import fitz


# ============================================================
# 1. AirWay.xlsx 내용을 코드 안에 심어 둔 기준 항공로 목록
#    형식: (진입지점, 항공로, 이탈지점, 참고)
# ============================================================
ROUTE_MASTER: List[Tuple[str, str, str, str]] = [
    ('AGAVO', 'GONAV - Y644 - EGOBA - Y697(G597) - KAE - Y437(B467)', 'KANSU', ''),
    ('AGAVO', 'GONAV - Y644 - EGOBA - Y697(G597) - KAE - Y437(B467) - TENAS - L512', 'ANDOL', ''),
    ('AGAVO', 'GONAV - Y644 - EGOBA - Y697(G597)', 'LANAT', ''),
    ('AGAVO', 'GONAV - Y644 - MONSI - Z54 - SOT - Y782(A582) - PSN - Z84', 'KALEK', ''),
    ('AGAVO', 'GONAV - Y644 - MONSI - Z54 - SOT - Y782 - TGU - Y781', 'BESNA', ''),
    ('AGAVO', 'GONAV - SOT - A582 - TGU - PSN', 'APELA', '담당자에게 문의 후 신청 (BESNA 활용 권고)'),
    ('AGAVO', 'GONAV - Y644 - MONSI - Y711 - KIDOS - Z81 - CJU - Y572(A586)', 'RUGMA', ''),
    ('AGAVO', 'RILRO - Z57 - DALPO - Y655 - TOLIS - Y655 - CJU - Y572(A586)', 'RUGMA', '수필 메모: Y677'),
    ('AGAVO', 'GONAV - Y644 - MONSI - Y711 - IKEDO - Y590', 'BEDAR', ''),
    ('AGAVO', 'RILRO - Z57 - DALPO - Y655 - TOLIS - Y655 - REMOS - Y711 - IKEDO - Y590', 'BEDAR', ''),
    ('AGAVO', 'GONAV - Y644 - MONSI - Y711 - PONIK - A593', 'LAMEN', ''),
    ('AGAVO', 'RILRO - Z57 - DALPO - Y655 - TOLIS - Y655 - REMOS - Y711 - PONIK - A593', 'LAMEN', ''),
    ('AGAVO', 'GONAV - Y644 - MONSI - Y711', 'MUGUS', ''),
    ('AGAVO', 'RILRO - Z57 - DALPO - Y655 - TOLIS - Y655 - REMOS - Y711', 'MUGUS', '수필 메모: Y677'),
    ('LAMEN', 'A593 - SADLI - Y590 - ELGEP - Y722 - SOSDO - Y571 - PSN - Y579 - TENAS - Y437(B467)', 'KANSU', ''),
    ('LAMEN', 'A593 - SADLI - Y590 - ELGEP - Y722 - SOT - Z54 - EGOBA - Y697(G597) - KAE - Y437(B467)', 'KANSU', 'A586/Y579 운항시간 외 이용 항공로'),
    ('LAMEN', 'A593 - SADLI - Y590 - ELGEP - Y722 - SOT - Z54 - Y697', 'AGAVO', ''),
    ('LAMEN', 'A593 - SADLI - Y590 - ELGEP - Y722 - CJU - Y655 - TOLIS - Y655 - NONOS - Z55', 'AGAVO', ''),
    ('LAMEN', 'A593 - SADLI - Y590 - ELGEP - Y722 - SOSDO - Y571 - PSN - Z84', 'KALEK', ''),
    ('LAMEN', 'A593 - SADLI - Y590 - ELGEP - Y722 - CJU - Y677(A595)', 'SAMDO', ''),
    ('ONIKU', 'A593 - NIRAT - Y722 - CJU - Y722 - SOT - Z54 - GOGET - Y697 - NOGON', 'AGAVO', ''),
    ('ONIKU', 'A593 - NIRAT - Y722 - CJU - Y655 - TOLIS - Y655 - NONOS - Z55', 'AGAVO', '수필 메모: Y677'),
    ('KANSU', 'Y437(B467) - TENAS - Y579(A586) - PSN - Y572 - CJU - Y711', 'MUGUS', '위에 언급된 A586/Y579 운항 가능시간과 동일'),
    ('KANSU', 'Y233 - BUSKO - Y437(B467) - KAE - Y697(G597) - EGOBA - Z50 - BULTI - Y711', 'MUGUS', 'Y579/A586 운항시간 외 이용 항공로'),
    ('KANSU', 'Y437(B467) - TENAS - Y579(A586) - PSN - Y572 - CJU - Z82 - PANSI - Y711 - PONIK - A593', 'LAMEN', 'A586/Y579 운항 가능시간 적용'),
    ('KANSU', 'Y233 - BUSKO - Y437(B467) - KAE - Y697(G597) - EGOBA - Z50 - BULTI - Y711 - PONIK - A593', 'LAMEN', 'Y579/A586 운항시간 외 이용 항공로'),
    ('KANSU', 'Y437(B467) - TENAS - Y579(A586) - PSN - Y572 - CJU - Y572(A586)', 'RUGMA', 'A586/Y579 운항 가능시간 적용'),
    ('KANSU', 'Y233 - BUSKO - Y437(B467) - KAE - Y697(G597) - EGOBA - Z50 - BULTI - Y711 - KIDOS - Z81 - CJU - Y572(A586)', 'RUGMA', 'Y579/A586 운항시간 외 이용 항공로'),
    ('ANDOL', 'L512 - TENAS - Y437(B467) - KAE - G597', 'AGAVO', 'RNAV 항공로 사용 불가 항공기 항공로'),
    ('ANDOL', 'L512 - TENAS - Y437(B467) - KAE - Y697(G597)', 'AGAVO', ''),
    ('SAPRA', 'Y685(G585) - SEL - Y697', 'AGAVO', ''),
    ('SAPRA', 'Y685(G585) - KPO - Y659(V549) - TGU - Z83 - TOPAX - CJU - Z82 - PANSI - Y711 - PONIK - A593', 'LAMEN', ''),
    ('SAPRA', 'Y685(G585) - KPO - Y659(V549) - TGU - Z83 - TOPAX - CJU - Z82 - PANSI - Y711', 'MUGUS', ''),
    ('KALEK', 'Z84 - PSN - Y572 - CJU - Z82 - PANSI - Y711', 'MUGUS', ''),
    ('KALEK', 'Z84 - PSN - Y572 - CJU - Z82 - PANSI - Y711 - PONIK - A593', 'LAMEN', ''),
    ('INVOK', 'Z91(G339) - PSN - Y782(A582) - BITUX - Z53 - BASEM - Y685(G585) - SEL - G597', 'AGAVO', 'RNAV 항공로 사용 불가 항공기 항공로'),
    ('APELA', 'Y782(A582) - PSN - A582 - TGU - A582 - BITUX - BASEM - G585 - SEL - G597', 'AGAVO', ''),
    ('INVOK', 'Z91(G339) - PSN - Y782(A582) - BITUX - Z53 - BASEM - Y685(G585) - SEL - Y697', 'AGAVO', ''),
    ('APELA', 'Y782(A582) - PSN - A582 - TGU - A582 - BITUX - BASEM - G585 - SEL - Y697', 'AGAVO', ''),
    ('SAMDO', 'Y677(A595) - CJU - Z82 - PANSI - Y711 - PONIK - A593', 'LAMEN', ''),
    ('SAMDO', 'Y677(A595) - CJU - Z82 - PANSI - Y711', 'MUGUS', ''),
    ('RUGMA', 'Y579(A586) - CJU - Y571 - PSN - Y579(A586) - TENAS - B467', 'KANSU', 'A586/Y579 운항 가능시간 적용'),
    ('RUGMA', 'Z85 - PAPLU - Y571 - PSN - Y579(A586) - TENAS - B467', 'KANSU', '담당자에게 문의 후 신청'),
    ('RUGMA', 'Y572(A586) - CJU - Y722 - SOT - Z54 - EGOBA - Y697 - KAE - B467', 'KANSU', 'Y579/A586 운항시간 외 이용 항공로'),
    ('RUGMA', 'Y572(A586) - CJU - Y722 - SOT - Z54 - GOGET - Y697 - NOGON', 'AGAVO', ''),
    ('RUGMA', 'Y572(A586) - CJU - Y655 - TOLIS - NONOS - Z55', 'AGAVO', '수필 메모: Y677'),
    ('ATOTI', 'Y722 - CJU - Y677(A595)', 'SAMDO', ''),
    ('ATOTI', 'Y722 - CJU - Y571 - PSN - Z84', 'KALEK', ''),
    ('ATOTI', 'Y722 - SOT - Z54 - GOGET - Y597', 'AGAVO', ''),
    ('ATOTI', 'Y722 - CJU - Y655 - TOLIS - Y655 - NONOS - Z55', 'AGAVO', '수필 메모: CJU 밑에 X 표시 및 Y677'),
    ('ATOTI', 'Y722 - CJU - Y571 - PSN - Y579(A586) - TENAS - Y437(B467)', 'KANSU', ''),
    ('ATOTI', 'Y722 - SOT - Z54 - EGOBA - Y697(G597) - KAE - Y437(B467)', 'KANSU', 'Y579/A586 운항시간 외 이용 항공로'),
]

# 항공로 구명칭/신명칭 병행 표기 허용값
ALIASES: Dict[str, List[str]] = {
    'Y685': ['G585'], 'G585': ['Y685'],
    'Y697': ['G597'], 'G597': ['Y697'],
    'Y437': ['B467'], 'B467': ['Y437'],
    'Y782': ['A582'], 'A582': ['Y782'],
    'Y572': ['A586'], 'Y579': ['A586'], 'A586': ['Y572', 'Y579'],
    'Y677': ['A595'], 'A595': ['Y677'],
    'Y659': ['V549'], 'V549': ['Y659'],
    'Z91': ['G339'], 'G339': ['Z91'],
}

DAYS = ['MON', 'TUE', 'WED', 'THU', 'FRI', 'SAT', 'SUN']
DAY_HEADER_NORMALIZE = {'MUN': 'MON', 'MON': 'MON', 'TUE': 'TUE', 'WED': 'WED', 'THU': 'THU', 'FRI': 'FRI', 'SAT': 'SAT', 'SUN': 'SUN'}

@dataclass
class FlightRecord:
    source_file: str
    page_no: int
    aircraft_id: str
    operator: str
    applicant: str
    email: str
    from_date: str
    to_date: str
    marked_days: List[str]
    entry_point: str
    airway_text: str
    exit_point: str
    route_remark: str = ''


class PermitValidationError(Exception):
    pass


def clean_token(value: str) -> str:
    return re.sub(r'[^A-Z0-9]', '', (value or '').upper())




def normalize_space(value: str) -> str:
    """여러 줄/여러 칸 공백을 파일명과 보고서에 쓰기 좋은 한 칸 공백으로 정리합니다."""
    return re.sub(r'\s+', ' ', value or '').strip()


def tidy_display_text(value: str) -> str:
    """출력용 이름/회사명 공백을 정리합니다. 예: '(EXECAIRE )' -> '(EXECAIRE)'"""
    value = normalize_space(value)
    value = re.sub(r'\(\s+', '(', value)
    value = re.sub(r'\s+\)', ')', value)
    value = re.sub(r'\s+,', ',', value)
    # 민원서류 접수 도장이 표와 겹쳐 Operator 칸에 섞이는 경우 제거
    for stamp_word in ['접수번호', '접수일시', '처리기한', '처리과', '기록물', '등록번호', '항공교통조정과']:
        value = value.replace(stamp_word, ' ')
    # 좁은 표 칸에서 CHARTERS가 C / HARTERS로 갈라지는 국내 민원서식 보정
    value = re.sub(r'\bC\s+HARTERS\b', 'CHARTERS', value)
    # 국내 민원서식의 좁은 Operator 칸에서 ANA WINGS가 ANA WING / S로 갈라지는 경우 보정
    value = re.sub(r'\bANA\s+WING\s+S\b', 'ANA WINGS', value, flags=re.IGNORECASE)
    return value.strip()


def english_token_text(value: str) -> str:
    """
    PDF에 민원서류 접수 도장 등 한글 오버레이가 겹쳐 들어오는 경우가 있어,
    Entry/Exit처럼 영문 코드만 필요한 칸은 영문/숫자 토큰만 남깁니다.
    예: 'SAPRA 민원서류' -> 'SAPRA'
    """
    tokens = re.findall(r'[A-Za-z0-9]+', value or '')
    return ' '.join(tokens).strip()


def clean_route_cell_display(value: str) -> str:
    """
    Entry/Exit/Airway 칸 표시용 정리.
    국내 민원서식처럼 Y685(G / 585), S / EL, Y6 / 97로 갈라진 항공로 조각은
    기준 토큰을 활용해 보기 좋게 복원합니다. 검증은 별도로 clean_token 기반으로 수행합니다.
    """
    cleaned = english_token_text(value)
    if cleaned:
        try:
            tokens = text_tokens(cleaned)
            if tokens:
                return ' '.join(tokens)
        except NameError:
            # text_tokens는 아래에서 정의됩니다. 모듈 초기화 중 직접 호출되는 경우를 대비한 안전장치입니다.
            pass
        return cleaned
    return normalize_space(value)

def known_route_tokens() -> set:
    """항공로 셀에서 줄바꿈 때문에 쪼개진 코드 조각을 복원하기 위한 기준 토큰 집합입니다."""
    tokens = set()
    for entry, route, exit_, _remark in ROUTE_MASTER:
        tokens.add(clean_token(entry))
        tokens.add(clean_token(exit_))
        for raw in re.findall(r'[A-Za-z0-9]+', route or ''):
            token = clean_token(raw)
            if token:
                tokens.add(token)
    for key, values in ALIASES.items():
        tokens.add(clean_token(key))
        for value in values:
            tokens.add(clean_token(value))
    return {t for t in tokens if t}


KNOWN_ROUTE_TOKENS = known_route_tokens()


def merge_fragmented_route_tokens(tokens: Sequence[str]) -> List[str]:
    """
    PDF 표 칸이 좁아 GONAV -> GONA/V, Y644 -> Y64/4처럼 잘리는 경우를 복원합니다.
    최대 4개 조각까지 붙여 기준 항공로 토큰과 일치하면 하나의 토큰으로 합칩니다.
    """
    cleaned = [clean_token(t) for t in tokens if clean_token(t)]
    merged: List[str] = []
    i = 0
    while i < len(cleaned):
        best = cleaned[i]
        best_j = i + 1
        max_j = min(len(cleaned), i + 4)
        for j in range(i + 1, max_j + 1):
            candidate = ''.join(cleaned[i:j])
            if candidate in KNOWN_ROUTE_TOKENS and len(candidate) > len(best):
                best = candidate
                best_j = j
        merged.append(best)
        i = best_j
    return merged


def collapse_adjacent_alias_tokens(tokens: Sequence[str]) -> List[str]:
    """Y697(G597)처럼 같은 항공로의 신/구 명칭이 같이 적힌 경우 한 토큰으로 봅니다."""
    collapsed: List[str] = []
    i = 0
    while i < len(tokens):
        current = clean_token(tokens[i])
        if i + 1 < len(tokens):
            nxt = clean_token(tokens[i + 1])
            if nxt in ALIASES.get(current, []) or current in ALIASES.get(nxt, []):
                collapsed.append(current)
                i += 2
                continue
        collapsed.append(current)
        i += 1
    return collapsed


def text_tokens(value: str) -> List[str]:
    raw_tokens = [clean_token(t) for t in re.findall(r'[A-Za-z0-9]+', value or '') if clean_token(t)]
    return merge_fragmented_route_tokens(raw_tokens)


def segment_alternatives(segment: str) -> set:
    tokens = text_tokens(segment)
    result = set(tokens)
    for token in list(result):
        result.update(ALIASES.get(token, []))
    return result


def master_route_segments(route_text: str) -> List[set]:
    segments = []
    for raw in route_text.split('-'):
        alt = segment_alternatives(raw)
        if alt:
            segments.append(alt)
    return segments


def normalize_user_airway_tokens(airway_text: str, entry: str, exit_: str) -> List[str]:
    tokens = text_tokens(airway_text)
    entry = clean_token(entry)
    exit_ = clean_token(exit_)

    # 신청서 항공로 칸에 진입/진출 지점을 함께 적는 경우가 있어 앞뒤에서 제거
    while tokens and tokens[0] == entry:
        tokens.pop(0)
    while tokens and tokens[-1] == exit_:
        tokens.pop()

    # Y697(G597)처럼 신/구 명칭을 괄호로 병기하면 PDF 추출 시 두 토큰으로 나뉘므로 하나로 접습니다.
    tokens = collapse_adjacent_alias_tokens(tokens)
    return tokens


def is_same_route(user_tokens: Sequence[str], master_segments: Sequence[set]) -> bool:
    """
    신청 항공로와 기준 항공로를 비교합니다.

    기본 원칙은 항공로 토큰 수와 순서가 기준과 정확히 일치해야 합니다.
    다만 실제 신청서에서 기준 항공로의 첫 경유점만 생략하고
    Entry Point를 항공로 칸 맨 앞에 반복 기재하는 사례가 확인되어,
    아래 한 가지 경우만 제한적으로 허용합니다.

    예) 기준: GONAV - Y644 - EGOBA - Y697(G597)
        신청: AGAVO Y644 EGOBA G597 LANAT
        정규화 후 신청: Y644 EGOBA G597
        => 기준의 첫 경유점 GONAV 생략으로 보고 허용

    중간 경유점 생략은 허용하지 않습니다.
    """
    cleaned_user = [clean_token(token) for token in user_tokens]

    def _match(tokens: Sequence[str], segments: Sequence[set]) -> bool:
        return len(tokens) == len(segments) and all(
            clean_token(token) in allowed for token, allowed in zip(tokens, segments)
        )

    # 1) 완전 일치
    if _match(cleaned_user, master_segments):
        return True

    # 2) 기준 항공로의 첫 경유점만 생략된 경우 허용
    if len(master_segments) > 1 and _match(cleaned_user, master_segments[1:]):
        return True

    return False


def validate_route(entry: str, airway_text: str, exit_: str) -> Tuple[bool, str]:
    entry = clean_token(entry)
    exit_ = clean_token(exit_)
    user_tokens = normalize_user_airway_tokens(airway_text, entry, exit_)

    candidates = [r for r in ROUTE_MASTER if clean_token(r[0]) == entry and clean_token(r[2]) == exit_]
    if not candidates:
        return False, f'기준표에 없는 진입/진출 조합입니다: {entry} -> {exit_}'

    for _, master_route, _, remark in candidates:
        if is_same_route(user_tokens, master_route_segments(master_route)):
            return True, remark or ''

    expected = ' / '.join(route for _, route, _, _ in candidates[:5])
    if len(candidates) > 5:
        expected += f' / ... 외 {len(candidates)-5}건'
    return False, f'항공로 불일치: 신청={" - ".join(user_tokens)} | 기준 후보={expected}'


def parse_date_yyyymmdd(value: str):
    return datetime.strptime(value, '%Y%m%d').date()


def expected_weekdays(from_date: str, to_date: str) -> List[str]:
    start = parse_date_yyyymmdd(from_date)
    end = parse_date_yyyymmdd(to_date)
    if end < start:
        raise PermitValidationError(f'기간 오류: from({from_date})이 to({to_date})보다 늦습니다.')

    days = []
    current = start
    while current <= end:
        day = DAYS[current.weekday()]
        if day not in days:
            days.append(day)
        current += timedelta(days=1)
    return days


def validate_days(from_date: str, to_date: str, marked_days: Sequence[str]) -> Tuple[bool, str]:
    expected = expected_weekdays(from_date, to_date)
    marked = list(dict.fromkeys(marked_days))  # 순서 유지 중복 제거
    if set(expected) != set(marked):
        return False, f'날짜-요일 불일치: 기간 {from_date}~{to_date}의 실제 요일={expected}, 신청서 표시={marked}'
    return True, ''


def word_center(word) -> Tuple[float, float]:
    x0, y0, x1, y1 = word[:4]
    return ((x0 + x1) / 2, (y0 + y1) / 2)


def words_in_box(words, x0, x1, y0, y1):
    selected = []
    for w in words:
        cx, cy = word_center(w)
        if x0 <= cx <= x1 and y0 <= cy <= y1:
            selected.append(w)
    return selected


def join_words(words) -> str:
    if not words:
        return ''
    # y값을 약간 뭉개서 같은 줄 순서를 안정화
    ordered = sorted(words, key=lambda w: (round(((w[1] + w[3]) / 2) / 3) * 3, w[0]))
    return ' '.join(w[4] for w in ordered).strip()


def extract_email(text: str) -> str:
    emails = re.findall(r'[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}', text)
    if not emails:
        raise PermitValidationError('E-MAIL 주소를 찾지 못했습니다.')
    return emails[-1]


def extract_applicant(words, text: str) -> str:
    # 좌측 Applicant 라벨과 같은 줄 중앙의 값을 우선 사용합니다.
    # 일부 양식은 "Applicant Vladimir APOGEE FZCO Signature Vladimir"처럼
    # Signature 뒤에 서명자명이 다시 찍히므로 Applicant와 Signature 사이만 사용합니다.
    applicant_words = [w for w in words if w[4].strip().lower() == 'applicant']
    for aw in applicant_words:
        _, cy = word_center(aw)
        same_line = [w for w in words if abs(word_center(w)[1] - cy) <= 5]
        sig_words = [w for w in same_line if w[4].strip().lower() == 'signature']
        left = aw[2]
        right = min((sw[0] for sw in sig_words), default=730)
        value_words = [
            w for w in same_line
            if left < word_center(w)[0] < right
            and w[4].strip().lower() not in {'applicant', 'signature'}
        ]
        value = join_words(value_words)
        if value:
            return normalize_space(value)

    # 국내 민원서식: '신청인(Applicant)' 라벨 오른쪽 값을 사용합니다.
    applicant_label_words = [w for w in words if 'applicant' in w[4].strip().lower()]
    for aw in applicant_label_words:
        if aw[4].strip().lower() == 'applicant':
            continue
        _, cy = word_center(aw)
        same_line = [w for w in words if abs(word_center(w)[1] - cy) <= 5]
        sig_words = [w for w in same_line if ('signature' in w[4].strip().lower() or '서명' in w[4])]
        left = aw[2]
        right = min((sw[0] for sw in sig_words), default=730)
        value_words = [
            w for w in same_line
            if left < word_center(w)[0] < right
            and 'applicant' not in w[4].strip().lower()
            and 'signature' not in w[4].strip().lower()
            and '서명' not in w[4]
        ]
        value = join_words(value_words)
        if value:
            return normalize_space(value)

    # fallback: 전체 텍스트에서 Applicant ~ Signature 사이만 사용
    m = re.search(r'Applicant\s+(.+?)\s+Signature', text, flags=re.I | re.S)
    if m:
        value = normalize_space(m.group(1))
        if value:
            return value

    # 국내 민원서식 fallback: 신청인(Applicant) ~ 서명 또는 인(Signature)
    m = re.search(r'신청인\s*\(Applicant\)\s+(.+?)\s+서명', text, flags=re.I | re.S)
    if m:
        value = normalize_space(m.group(1))
        if value:
            return value
    raise PermitValidationError('Applicant 정보를 찾지 못했습니다.')

def final_applicant_by_email(applicant: str, email: str) -> str:
    domain = email.split('@')[-1].strip().lower() if '@' in email else ''
    if domain == 'jetex.com' or domain.endswith('.jetex.com'):
        return 'JETEX FLIGHT SUPPORT'
    return applicant


def day_header_centers(words) -> Dict[str, float]:
    centers: Dict[str, float] = {}

    # 1) 일반 양식: MON/TUE/WED... 또는 MUN/TUE/WED...가 한 단어로 추출되는 경우
    for w in words:
        token = w[4].strip().upper()
        if token in DAY_HEADER_NORMALIZE:
            cx, cy = word_center(w)
            # 표 상단 요일 헤더 영역만 사용
            if 75 <= cy <= 170:
                centers[DAY_HEADER_NORMALIZE[token]] = cx

    # 2) 세로쓰기 양식: M/O/N, T/U/E처럼 흩어지고, 아래 1~7 숫자로 열이 표시되는 경우
    #    이 경우 1~7 숫자 중심을 MON~SUN 중심으로 사용합니다.
    numeric_centers: Dict[str, float] = {}
    for w in words:
        token = w[4].strip()
        cx, cy = word_center(w)
        # 1) CGHSW류: 1~7 숫자가 표 상단 145~180 부근
        # 2) 국내 민원서식: 1~7 숫자가 300~320 부근에 배치됨
        in_supported_band = (300 <= cx <= 520 and 145 <= cy <= 180) or (260 <= cx <= 370 and 295 <= cy <= 325)
        if token in {'1', '2', '3', '4', '5', '6', '7'} and in_supported_band:
            numeric_centers[DAYS[int(token) - 1]] = cx
    if len(numeric_centers) >= 5:
        centers.update(numeric_centers)

    return centers


def marked_days_for_row(words, y0, y1, fallback_centers: Optional[Dict[str, float]] = None) -> List[str]:
    centers = day_header_centers(words)
    if not centers and fallback_centers:
        centers = fallback_centers
    if not centers:
        raise PermitValidationError('요일 헤더를 찾지 못했습니다.')

    x_min = min(centers.values()) - 25
    x_max = max(centers.values()) + 25
    marks = []
    for w in words:
        raw_original = w[4].strip().upper()
        raw = raw_original.replace('\u3000', ' ')
        cx, cy = word_center(w)
        if not (x_min <= cx <= x_max and y0 <= cy <= y1):
            continue

        # 일반 양식: X, V, 체크마크 등
        if raw in {'X', 'V', '√', '✓', '✔', '☑', '■'}:
            nearest_day = min(centers, key=lambda d: abs(centers[d] - cx))
            marks.append(nearest_day)
            continue

        # 정기편 양식: 한 셀/한 단어에 '1 2'처럼 여러 운항요일 숫자가 붙어 나오는 경우
        # 예: DAL389류에서 MON/TUE가 '1　2' 한 토큰으로 추출됩니다.
        digit_list = re.findall(r'[1-7]', raw)
        non_digit = re.sub(r'[1-7\s]', '', raw)
        if digit_list and not non_digit:
            for d in digit_list:
                marks.append(DAYS[int(d) - 1])
            continue

        # 세로쓰기/숫자 체크 양식: 해당 요일 칸에 1~7 숫자가 직접 들어가는 경우
        digits = re.sub(r'[^0-9]', '', raw)
        if digits in {'1', '2', '3', '4', '5', '6', '7'}:
            day = DAYS[int(digits) - 1]
            # 숫자가 자기 요일 열 근처에 있을 때만 체크 표시로 인정합니다.
            if day in centers and abs(cx - centers[day]) <= 22:
                marks.append(day)
                continue

    return list(dict.fromkeys(marks))

def merged_date_words(words) -> List[Tuple[float, float, float, float, str]]:
    """
    일부 PDF는 날짜가 2026050 / 4 처럼 같은 칸에서 두 줄로 쪼개져 추출됩니다.
    그런 경우 같은 x축·가까운 y축의 숫자 토큰을 합쳐 8자리 날짜 토큰으로 복원합니다.
    """
    direct = [(w[0], w[1], w[2], w[3], w[4]) for w in words if re.fullmatch(r'\d{8}', w[4])]
    merged: List[Tuple[float, float, float, float, str]] = []

    numeric_words = [w for w in words if re.fullmatch(r'\d{1,8}', w[4])]
    for w in numeric_words:
        token = w[4]
        if len(token) >= 8:
            continue
        x0, y0, x1, y1 = w[:4]
        cx, cy = word_center(w)
        # 같은 칸에서 아래쪽에 이어지는 숫자 조각을 찾습니다.
        candidates = []
        for w2 in numeric_words:
            if w2 is w:
                continue
            token2 = w2[4]
            cx2, cy2 = word_center(w2)
            if abs(cx2 - cx) <= 18 and 0 < cy2 - cy <= 16:
                combined = token + token2
                if re.fullmatch(r'\d{8}', combined) and combined.startswith(('19', '20')):
                    candidates.append((cy2, w2, combined))
        if candidates:
            _, w2, combined = sorted(candidates, key=lambda x: x[0])[0]
            mx0 = min(w[0], w2[0]); my0 = min(w[1], w2[1]); mx1 = max(w[2], w2[2]); my1 = max(w[3], w2[3])
            merged.append((mx0, my0, mx1, my1, combined))

    # 중복 제거: 같은 날짜/거의 같은 x,y는 하나만 남깁니다.
    all_dates = direct + merged
    unique = []
    seen = set()
    for item in all_dates:
        key = (item[4], round(((item[0] + item[2]) / 2) / 2) * 2, round(((item[1] + item[3]) / 2) / 2) * 2)
        if key not in seen:
            seen.add(key)
            unique.append(item)
    return unique


def layout_boxes_for_row(from_date_word, words=None) -> Dict[str, Tuple[float, float]]:
    """신청서 표 양식이 여러 종류라 날짜 칸 위치와 페이지 폭으로 레이아웃을 판별합니다."""
    from_x, _ = word_center(from_date_word)
    max_x = max((w[2] for w in words), default=800) if words is not None else 800

    if max_x < 700:
        # 국내 민원서식 세로형: 092_1_1_81.pdf류
        return {
            'operator': (72, 118),
            'aircraft': (120, 158),
            'entry': (410, 443),
            'airway': (464, 498),
            'exit': (500, 530),
        }

    if from_x > 260:
        # CGHSW류: Period of overflight가 가운데에 배치된 양식
        return {
            'operator': (95, 155),
            'aircraft': (160, 212),
            'entry': (588, 628),
            'airway': (672, 716),
            'exit': (724, 760),
        }
    # 기존 샘플류
    return {
        'operator': (67, 113),
        'aircraft': (112, 155),
        'entry': (620, 657),
        'airway': (690, 739),
        'exit': (738, 776),
    }

def extract_flight_records_from_page(words, source_file: str, page_no: int, applicant: str, email: str, day_centers_fallback: Optional[Dict[str, float]] = None) -> List[FlightRecord]:
    # 날짜 단어가 있는 행을 신청 행으로 판단합니다.
    # 날짜가 2026050 / 4처럼 줄바꿈되어 추출되는 양식도 함께 복원합니다.
    date_words = merged_date_words(words)
    clusters: List[List] = []
    for w in sorted(date_words, key=lambda w: word_center(w)[1]):
        _, cy = word_center(w)
        placed = False
        for cluster in clusters:
            if abs(word_center(cluster[0])[1] - cy) <= 8:
                cluster.append(w)
                placed = True
                break
        if not placed:
            clusters.append([w])

    # 행 간격이 촘촘한 정기편/다건 신청서에서는 ±28 고정폭이 위아래 행을 침범할 수 있으므로
    # 날짜 클러스터 중심 사이의 중간값으로 행 경계를 계산합니다.
    row_items = []
    for cluster in clusters:
        if len(cluster) < 2:
            continue
        cluster_sorted = sorted(cluster, key=lambda w: w[0])
        cy_values = [word_center(item)[1] for item in cluster_sorted[:2]]
        row_items.append((sum(cy_values) / len(cy_values), cluster_sorted))
    row_items.sort(key=lambda item: item[0])

    records = []
    for idx, (row_cy, cluster_sorted) in enumerate(row_items):
        from_date, to_date = cluster_sorted[0][4], cluster_sorted[1][4]
        _, cy = word_center(cluster_sorted[0])

        # 기본 행 높이. 행이 3개 이상이면 인접 행과의 중간선을 경계로 삼아
        # DAL389류처럼 표가 촘촘한 경우 항공로가 서로 섞이지 않게 합니다.
        y0, y1 = cy - 28, cy + 28
        if len(row_items) >= 3:
            if idx > 0:
                y0 = (row_items[idx - 1][0] + row_cy) / 2
            else:
                next_gap = row_items[idx + 1][0] - row_cy
                y0 = row_cy - max(28, next_gap / 2)
            if idx + 1 < len(row_items):
                y1 = (row_cy + row_items[idx + 1][0]) / 2
            else:
                prev_gap = row_cy - row_items[idx - 1][0]
                y1 = row_cy + max(28, prev_gap / 2)

        boxes = layout_boxes_for_row(cluster_sorted[0], words)
        operator = join_words(words_in_box(words, *boxes['operator'], y0, y1))
        aircraft_id = join_words(words_in_box(words, *boxes['aircraft'], y0, y1))
        entry_point = clean_route_cell_display(join_words(words_in_box(words, *boxes['entry'], y0, y1)))
        airway_text = clean_route_cell_display(join_words(words_in_box(words, *boxes['airway'], y0, y1)))
        exit_point = clean_route_cell_display(join_words(words_in_box(words, *boxes['exit'], y0, y1)))
        marked_days = marked_days_for_row(words, y0, y1, day_centers_fallback)

        missing = []
        if not operator:
            missing.append('Operator')
        if not aircraft_id:
            missing.append('Aircraft identification')
        if not entry_point:
            missing.append('Incheon FIR Entry Point')
        if not airway_text:
            missing.append('Air-way')
        if not exit_point:
            missing.append('Incheon FIR Exit Point')
        if missing:
            raise PermitValidationError(f'{source_file} {page_no}페이지 신청 행 필수값 누락: {", ".join(missing)}')

        records.append(FlightRecord(
            source_file=source_file,
            page_no=page_no,
            aircraft_id=aircraft_id,
            operator=operator,
            applicant=applicant,
            email=email,
            from_date=from_date,
            to_date=to_date,
            marked_days=marked_days,
            entry_point=entry_point,
            airway_text=airway_text,
            exit_point=exit_point,
        ))
    return records


def rect_intersects(a, b, padding=1.0) -> bool:
    return not (a.x1 < b.x0 - padding or a.x0 > b.x1 + padding or a.y1 < b.y0 - padding or a.y0 > b.y1 + padding)


def validate_dangerous_goods_no(page) -> Tuple[bool, str]:
    words = page.get_text('words')
    dg_candidates = [w for w in words if w[4].strip().lower().startswith('dangerous')]
    if not dg_candidates:
        return False, '위험물(Dangerous Goods) 항목을 찾지 못했습니다.'

    # Freight List 아래 '- Armament or Dangerous Goods' 라인을 사용합니다.
    # 페이지 하단 안내문(If Dangerous Goods are boarded...)의 Dangerous는 제외합니다.
    dg_word = next((w for w in dg_candidates if word_center(w)[0] < 220), dg_candidates[0])
    _, dg_y = word_center(dg_word)
    area_y0, area_y1 = dg_y - 18, dg_y + 55
    area_x0, area_x1 = 15, 285

    drawings = page.get_drawings()
    rects = []
    line_rects = []
    for d in drawings:
        rect = d.get('rect')
        if rect is None:
            continue
        cx = (rect.x0 + rect.x1) / 2
        cy = (rect.y0 + rect.y1) / 2
        w = rect.x1 - rect.x0
        h = rect.y1 - rect.y0
        if not (area_x0 <= cx <= area_x1 and area_y0 <= cy <= area_y1):
            continue
        if 2.5 <= w <= 10 and 2.5 <= h <= 10:
            # 작은 사각형: 체크박스 후보
            if any(item[0] == 're' for item in d.get('items', [])):
                rects.append(rect)
        # 체크 표시 또는 사각형 내부 대각선 후보. 일부 PDF는 체크 표시만 선분으로 렌더링됩니다.
        if d.get('type') == 's' and 1.5 <= max(w, h) <= 12 and 1.0 <= min(w if w else 1.0, h if h else 1.0) <= 12:
            if any(item[0] == 'l' for item in d.get('items', [])):
                line_rects.append(rect)

    yes_words = [w for w in words if w[4].strip().lower().startswith('yes') and abs(word_center(w)[1] - dg_y) <= 14]
    no_words = [w for w in words if w[4].strip().lower() == 'no' and area_y0 <= word_center(w)[1] <= area_y1]

    # 국내 민원서식처럼 체크박스 없이 'Dangerous Goods) : NO'로 직접 기재된 경우
    # 기존 영문 양식의 'Dangerous Goods : Yes , No'는 선택지 나열이므로 여기서 판정하지 않습니다.
    text = page.get_text('text')
    lines = text.splitlines()
    for i, line in enumerate(lines):
        if re.search(r'Dangerous\s+Goods', line, flags=re.I) and '탑재여부' in line:
            next_line = lines[i + 1] if i + 1 < len(lines) else ''
            merged_line = f'{line} {next_line}'.upper()
            has_yes = re.search(r'\bYES\b', merged_line) is not None
            has_no = re.search(r'\bNO\b', merged_line) is not None
            if has_no and not has_yes:
                return True, ''
            if has_yes and not has_no:
                return False, '위험물 항목이 Yes로 기재되어 있습니다.'

    # 영문 일반 양식 중 체크박스가 없거나 추출이 불안정하고, 선택지가 아니라 'No'만 직접 기재된 경우
    # 예: '- Armament or Dangerous Goods : No □'
    for i, line in enumerate(lines):
        if re.search(r'Armament\s+or\s+Dangerous\s+Goods|Dangerous\s+Goods\)?', line, flags=re.I):
            next_line = lines[i + 1] if i + 1 < len(lines) else ''
            merged_line = f'{line} {next_line}'.upper()
            has_yes = re.search(r'\bYES\b', merged_line) is not None
            has_no = re.search(r'\bNO\b', merged_line) is not None
            if has_no and not has_yes:
                return True, ''
            if has_yes and not has_no:
                return False, '위험물 항목이 Yes로 기재되어 있습니다.'

    def box_center(rect):
        return ((rect.x0 + rect.x1) / 2, (rect.y0 + rect.y1) / 2)

    def is_checked(box):
        return sum(1 for lr in line_rects if rect_intersects(box, lr, padding=1.8)) >= 1

    # 1) 도형 사각형 체크박스가 있는 일반 양식
    if rects:
        yes_box = None
        if yes_words:
            yes_word = yes_words[-1]
            yes_x1 = yes_word[2]
            candidates = [r for r in rects if yes_x1 <= box_center(r)[0] <= yes_x1 + 45 and abs(box_center(r)[1] - word_center(yes_word)[1]) <= 15]
            if candidates:
                yes_box = min(candidates, key=lambda r: abs(box_center(r)[0] - yes_x1))

        yes_checked = bool(yes_box and is_checked(yes_box))
        checked_boxes = [r for r in rects if is_checked(r)]
        no_checked = any(r is not yes_box for r in checked_boxes)

        if yes_checked:
            return False, '위험물 항목이 Yes로 체크되어 있습니다.'
        if no_checked:
            return True, ''

        if no_words:
            no_word = no_words[-1]
            no_x0, no_x1 = no_word[0], no_word[2]
            no_candidates = [r for r in rects if no_x0 - 35 <= box_center(r)[0] <= no_x1 + 45 and abs(box_center(r)[1] - word_center(no_word)[1]) <= 15]
            if any(is_checked(r) for r in no_candidates):
                return True, ''

    # 2) 문자 '□'와 체크 선분만 추출되는 양식(CGHSW류)
    #    Yes 라인에 선분이 있으면 Yes 체크, No 라인/No 박스 근처에 선분이 있으면 No 체크로 판단합니다.
    yes_checked_by_line = False
    if yes_words:
        yw = yes_words[-1]
        ycx, ycy = word_center(yw)
        for lr in line_rects:
            cx = (lr.x0 + lr.x1) / 2; cy = (lr.y0 + lr.y1) / 2
            if yw[2] <= cx <= yw[2] + 35 and abs(cy - ycy) <= 12:
                yes_checked_by_line = True
                break

    no_checked_by_line = False
    if no_words:
        nw = no_words[-1]
        ncx, ncy = word_center(nw)
        for lr in line_rects:
            cx = (lr.x0 + lr.x1) / 2; cy = (lr.y0 + lr.y1) / 2
            if nw[0] <= cx <= nw[2] + 25 and abs(cy - ncy) <= 12:
                no_checked_by_line = True
                break

    if yes_checked_by_line:
        return False, '위험물 항목이 Yes로 체크되어 있습니다.'
    if no_checked_by_line:
        return True, ''

    if not rects and not line_rects:
        return False, '위험물 Yes/No 체크박스를 찾지 못했습니다.'

    return False, '위험물 항목이 No로 체크되어 있지 않습니다.'

def validate_dangerous_goods_no_in_doc(doc, source_file: str) -> List[str]:
    """
    2페이지 이상 신청서 대응용 위험물 검증.
    - 위험물 체크란은 보통 마지막 페이지나 첫 페이지에 1회만 나타납니다.
    - 따라서 모든 페이지에 체크란이 있다고 가정하지 않습니다.
    - Armament/Freight List/Dangerous Goods: Yes 문구가 있는 후보 페이지만 검사합니다.
    """
    candidate_errors = []
    checked_relevant_page = False

    for page_index in range(len(doc)):
        page = doc[page_index]
        page_no = page_index + 1
        text = page.get_text('text')

        # 표만 이어지는 페이지에는 위험물 체크란이 없을 수 있으므로 건너뜁니다.
        is_candidate = bool(re.search(
            # 안내문(detail Freight List)만 있는 서명 페이지는 제외하고,
            # 실제 위험물 체크/기재 행이 있는 페이지만 후보로 삼습니다.
            r'Armament\s+or\s+Dangerous\s+Goods|Dangerous\s+Goods\)?\s*:\s*(Yes|No)|Freight\s+List\s*:|위험물.*탑재여부',
            text,
            flags=re.I | re.S
        ))
        if not is_candidate:
            continue

        checked_relevant_page = True
        dg_ok, dg_msg = validate_dangerous_goods_no(page)
        if dg_ok:
            return []
        candidate_errors.append(f'{source_file} {page_no}페이지: {dg_msg}')

    if not checked_relevant_page:
        return [f'{source_file}: 위험물(Dangerous Goods) 항목을 찾지 못했습니다.']

    return candidate_errors or [f'{source_file}: 위험물 항목이 No로 체크되어 있지 않습니다.']


def extract_global_email_and_applicant(doc, source_file: str) -> Tuple[str, str]:
    """
    2페이지 이상 신청서 대응용 공통 정보 추출.
    E-MAIL과 Applicant가 특정 페이지에만 있어도 전체 문서에서 찾습니다.
    jetex.com 도메인은 Applicant를 JETEX FLIGHT SUPPORT로 강제합니다.
    """
    all_text_parts = []
    page_payloads = []

    for page_index in range(len(doc)):
        page = doc[page_index]
        text = page.get_text('text')
        words = page.get_text('words')
        all_text_parts.append(text)
        page_payloads.append((page_index + 1, words, text))

    all_text = '\n'.join(all_text_parts)
    email = extract_email(all_text)

    domain = email.split('@')[-1].strip().lower() if '@' in email else ''
    if domain == 'jetex.com' or domain.endswith('.jetex.com'):
        return email, 'JETEX FLIGHT SUPPORT'

    applicant_errors = []
    for page_no, words, text in page_payloads:
        try:
            applicant = extract_applicant(words, text)
            return email, applicant
        except Exception as e:
            applicant_errors.append(f'{page_no}페이지: {e}')

    # 최후 보조: 전체 텍스트에서 Applicant ~ Signature 패턴 탐색
    m = re.search(r'Applicant\s+(.+?)\s+Signature', all_text, flags=re.I | re.S)
    if m:
        value = re.sub(r'\s+', ' ', m.group(1)).strip()
        if value:
            return email, value

    raise PermitValidationError(f'{source_file}: Applicant 정보를 찾지 못했습니다.')



def repair_cross_row_airway_bleed(records: List[FlightRecord]) -> None:
    """
    표가 매우 촘촘한 2페이지 이상 정기편 신청서에서 항공로 마지막 줄이 다음 행으로
    붙어 추출되는 경우를 보정합니다. 예: SAPRA->MUGUS 행의 'PANSI Y711'이
    다음 ATOTI->KALEK 행 앞에 붙는 DAL389류 PDF.
    """
    for idx in range(len(records) - 1):
        current = records[idx]
        nxt = records[idx + 1]
        if current.source_file != nxt.source_file or current.page_no != nxt.page_no:
            continue

        current_tokens = normalize_user_airway_tokens(current.airway_text, current.entry_point, current.exit_point)
        next_tokens = normalize_user_airway_tokens(nxt.airway_text, nxt.entry_point, nxt.exit_point)
        if len(next_tokens) < 3:
            continue

        # 현재 행의 기준 후보 중 마지막 부분이 PANSI-Y711로 끝나는데 현재 추출값에는 없고,
        # 다음 행 맨 앞에 그 두 토큰이 붙어 있으면 현재 행으로 되돌립니다.
        candidates = [r for r in ROUTE_MASTER if clean_token(r[0]) == clean_token(current.entry_point) and clean_token(r[2]) == clean_token(current.exit_point)]
        needs_pansi_y711 = any(
            len(master_route_segments(route)) >= 2
            and 'PANSI' in master_route_segments(route)[-2]
            and 'Y711' in master_route_segments(route)[-1]
            for _, route, _, _ in candidates
        )
        has_pansi_y711 = any(t == 'PANSI' for t in current_tokens) and any(t == 'Y711' for t in current_tokens)
        next_starts_with_pansi_y711 = len(next_tokens) >= 2 and next_tokens[0] == 'PANSI' and next_tokens[1] == 'Y711'

        if needs_pansi_y711 and not has_pansi_y711 and next_starts_with_pansi_y711:
            current.airway_text = normalize_space(current.airway_text + ' PANSI Y711')
            nxt.airway_text = ' '.join(next_tokens[2:])

def extract_records_from_pdf(pdf_path: str) -> List[FlightRecord]:
    doc = fitz.open(pdf_path)
    source_file = os.path.basename(pdf_path)
    all_records: List[FlightRecord] = []
    file_errors = []

    try:
        email, applicant = extract_global_email_and_applicant(doc, source_file)
    except Exception as e:
        file_errors.append(str(e))
        email, applicant = '', ''

    file_errors.extend(validate_dangerous_goods_no_in_doc(doc, source_file))

    last_day_centers: Optional[Dict[str, float]] = None

    for page_index in range(len(doc)):
        page = doc[page_index]
        page_no = page_index + 1
        words = page.get_text('words')

        page_day_centers = day_header_centers(words)
        if page_day_centers:
            last_day_centers = page_day_centers

        try:
            page_records = extract_flight_records_from_page(
                words,
                source_file,
                page_no,
                applicant,
                email,
                day_centers_fallback=last_day_centers,
            )
            all_records.extend(page_records)
        except Exception as e:
            # 날짜 행이 없는 안내/서명 페이지는 정상입니다.
            # 날짜 행이 있는데 필수값이 빠진 경우 등은 extract 함수 안에서 오류가 납니다.
            file_errors.append(f'{source_file} {page_no}페이지: {e}')

    repair_cross_row_airway_bleed(all_records)

    if not all_records:
        file_errors.append(f'{source_file}: 신청 행을 찾지 못했습니다.')

    if file_errors:
        raise PermitValidationError('\n'.join(file_errors))
    return all_records


def validate_record(record: FlightRecord) -> List[str]:
    errors = []
    ok_days, msg_days = validate_days(record.from_date, record.to_date, record.marked_days)
    if not ok_days:
        errors.append(f'{record.source_file} {record.page_no}페이지 {record.aircraft_id}: {msg_days}')

    ok_route, route_msg = validate_route(record.entry_point, record.airway_text, record.exit_point)
    if not ok_route:
        errors.append(f'{record.source_file} {record.page_no}페이지 {record.aircraft_id}: {route_msg}')
    else:
        record.route_remark = route_msg
    return errors


def format_output_block(record: FlightRecord) -> str:
    date_text = datetime.strptime(record.from_date, '%Y%m%d').strftime('%Y/%m/%d')
    application_info = f'{record.email}{date_text}{record.aircraft_id}'
    return (
        f'1. Aircraft identification : {record.aircraft_id}\n'
        f'2. Operator : {tidy_display_text(record.operator)}\n'
        f'3. Applicant : {tidy_display_text(record.applicant)}\n'
        f'4. 신청정보 : {application_info}'
    )


def format_record_summary(record: FlightRecord) -> str:
    """오류보고서에 넣을 신청 건 요약."""
    return (
        f'- 파일/페이지: {record.source_file} / {record.page_no}페이지\n'
        f'  Aircraft identification: {record.aircraft_id}\n'
        f'  Operator: {tidy_display_text(record.operator)}\n'
        f'  Applicant: {tidy_display_text(record.applicant)}\n'
        f'  E-MAIL: {record.email}\n'
        f'  기간: {record.from_date} ~ {record.to_date}\n'
        f'  표시 요일: {", ".join(record.marked_days) if record.marked_days else "없음"}\n'
        f'  Entry: {record.entry_point}\n'
        f'  Airway: {record.airway_text}\n'
        f'  Exit: {record.exit_point}'
    )


def write_error_report(
    errors: Sequence[str],
    pdf_paths: Sequence[str],
    records: Optional[Sequence[FlightRecord]] = None,
    report_path: str = '검증오류보고서.txt',
) -> str:
    """검증 실패 내용을 사람이 읽기 쉬운 txt로 저장합니다."""
    now_text = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    records = list(records or [])

    lines = [
        '대한민국 영공통과 허가 신청서 자동검증 오류보고서',
        '=' * 58,
        f'생성시각: {now_text}',
        '',
        '[처리 대상 파일]',
    ]
    for path in pdf_paths:
        lines.append(f'- {os.path.basename(path)}')

    lines.extend([
        '',
        '[검증 결과]',
        '검증 실패: 허가 신청 템플릿 파일을 생성하지 않았습니다.',
        '',
        '[오류 목록]',
    ])

    if errors:
        for idx, err in enumerate(errors, start=1):
            # 여러 줄 오류도 들여쓰기해 읽기 쉽게 정리
            err_lines = str(err).splitlines() or ['']
            lines.append(f'{idx}. {err_lines[0]}')
            for extra in err_lines[1:]:
                lines.append(f'   {extra}')
    else:
        lines.append('1. 구체적인 오류 메시지가 수집되지 않았습니다.')

    if records:
        lines.extend([
            '',
            '[추출된 신청 건 요약]',
            '※ 아래 정보는 PDF에서 읽어낸 값입니다. 오류 수정 시 참고용으로만 사용하세요.',
        ])
        for record in records:
            lines.append('')
            lines.append(format_record_summary(record))

    lines.extend([
        '',
        '[조치 안내]',
        '- 날짜와 요일 표시가 실제 달력과 일치하는지 확인하세요.',
        '- Entry Point, Airway, Exit Point 조합이 기준 항공로와 일치하는지 확인하세요.',
        '- Armament or Dangerous Goods 항목이 반드시 No로 체크되어 있는지 확인하세요.',
        '- E-MAIL, Applicant, Aircraft identification 값이 PDF에서 정상 추출되었는지 확인하세요.',
    ])

    with open(report_path, 'w', encoding='utf-8-sig') as f:
        f.write('\n'.join(lines))
    return report_path


def sanitize_filename_part(value: str) -> str:
    """Windows/Colab 다운로드에서 문제가 되는 파일명 문자를 정리합니다."""
    value = normalize_space(value or '')
    value = re.sub(r'[\\/:*?"<>|]+', '_', value)
    value = re.sub(r'\s+', ' ', value).strip(' .')
    return value or 'UNKNOWN'


def build_output_filename(records: Sequence[FlightRecord]) -> str:
    """
    Aircraft identification을 반영한 출력 파일명을 만듭니다.
    신청 건이 2건 이상이면 같은 항공기 번호가 반복되더라도
    첫 번째 Aircraft identification 기준으로 '외 n건' 형식을 사용합니다.
    """
    if not records:
        label = 'UNKNOWN'
    else:
        first_aircraft_id = sanitize_filename_part(records[0].aircraft_id)
        if len(records) == 1:
            label = first_aircraft_id
        else:
            label = f'{first_aircraft_id} 외 {len(records) - 1}건'

    return f'영공통과 허가 신청서({label}).txt'


def process_pdfs(
    pdf_paths: Sequence[str],
    output_path: str | None = None,
    error_report_path: str = '검증오류보고서.txt',
) -> Tuple[str, List[FlightRecord]]:
    records: List[FlightRecord] = []
    errors: List[str] = []

    for pdf_path in pdf_paths:
        try:
            records.extend(extract_records_from_pdf(pdf_path))
        except Exception as e:
            errors.append(str(e))

    for record in records:
        try:
            errors.extend(validate_record(record))
        except Exception as e:
            errors.append(f'{record.source_file} {record.page_no}페이지 {record.aircraft_id}: {e}')

    if not records:
        errors.append('처리할 신청 기록이 없습니다.')

    if errors:
        report_path = write_error_report(errors, pdf_paths, records, error_report_path)
        error_text = '\n'.join(f'- {err}' for err in errors)
        raise PermitValidationError(
            '검증 오류가 발생했습니다. 허가 신청 템플릿 파일을 생성하지 않았습니다.\n'
            f'오류보고서 파일: {report_path}\n'
            + error_text
        )

    if output_path is None:
        output_path = build_output_filename(records)

    content = '\n\n'.join(format_output_block(r) for r in records)
    with open(output_path, 'w', encoding='utf-8-sig') as f:
        f.write(content)
    return output_path, records


def run_colab_upload():
    try:
        from google.colab import files
    except Exception:
        files = None

    if files:
        print('PDF 신청서를 선택하세요. 여러 개를 한 번에 선택할 수 있습니다.')
        uploaded = files.upload()
        pdf_paths = [name for name in uploaded.keys() if name.lower().endswith('.pdf')]
        if not pdf_paths:
            report_path = write_error_report(['업로드한 파일 중 PDF가 없습니다.'], [], [])
            print('❌ 검증 실패: 업로드한 파일 중 PDF가 없습니다.')
            print(f'📄 오류보고서 생성: {report_path}')
            files.download(report_path)
            return
    else:
        path = input('PDF 파일 경로를 입력하세요: ').strip().strip('"')
        pdf_paths = [path]

    try:
        output_path, records = process_pdfs(pdf_paths)
    except PermitValidationError as e:
        report_path = '검증오류보고서.txt'
        print('❌ 검증 실패')
        print(str(e))
        if os.path.exists(report_path):
            print('\n--- 오류보고서 미리보기 ---')
            with open(report_path, 'r', encoding='utf-8-sig') as f:
                print(f.read())
            if files:
                files.download(report_path)
        return
    except Exception as e:
        # 예상하지 못한 예외도 Colab traceback으로 끝내지 않고 txt로 남깁니다.
        report_path = write_error_report([f'예상하지 못한 프로그램 오류: {e}'], pdf_paths, [])
        print('❌ 프로그램 오류가 발생했습니다.')
        print(f'📄 오류보고서 생성: {report_path}')
        if files:
            files.download(report_path)
        return

    print(f'✅ 검증 통과: {len(records)}건')
    print(f'✅ 생성 파일: {output_path}')
    print('\n--- 생성 내용 미리보기 ---')
    with open(output_path, 'r', encoding='utf-8-sig') as f:
        print(f.read())

    if files:
        files.download(output_path)


if __name__ == '__main__':
    run_colab_upload()


PDF 신청서를 선택하세요. 여러 개를 한 번에 선택할 수 있습니다.


Saving 영공통과 항행 허가 신청서(NJE052M).pdf to 영공통과 항행 허가 신청서(NJE052M) (1).pdf
✅ 검증 통과: 2건
✅ 생성 파일: 영공통과 허가 신청서(NJE052M 외 1건).txt

--- 생성 내용 미리보기 ---
1. Aircraft identification : NJE052M
2. Operator : Netjets Transportes Aereos SA
3. Applicant : JETEX FLIGHT SUPPORT
4. 신청정보 : a-team@jetex.com2026/05/13NJE052M

1. Aircraft identification : NJE052M
2. Operator : Netjets Transportes Aereos SA
3. Applicant : JETEX FLIGHT SUPPORT
4. 신청정보 : a-team@jetex.com2026/05/16NJE052M


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
run_colab_upload()

PDF 신청서를 선택하세요. 여러 개를 한 번에 선택할 수 있습니다.


Saving 영공통과 항행 허가 신청서(N318LE).pdf to 영공통과 항행 허가 신청서(N318LE).pdf


PermitValidationError: 검증 오류가 발생했습니다. 텍스트 파일을 생성하지 않습니다.
- 영공통과 항행 허가 신청서(N318LE).pdf 1페이지: 위험물 Yes/No 체크박스를 찾지 못했습니다.
영공통과 항행 허가 신청서(N318LE).pdf 1페이지: Applicant 정보를 찾지 못했습니다.
영공통과 항행 허가 신청서(N318LE).pdf: 신청 행을 찾지 못했습니다.